# Installation et import des libraries

In [1]:
%pip install tensorflow matplotlib scikit-learn tqdm

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.35.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached requests-2.34.2-py3-none-any.w

**Chargement des données**

In [2]:
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import collections
import random
import re
import numpy as np
import os
import time
import json
from glob import glob
from PIL import Image
import pickle
from tqdm import tqdm

I0000 00:00:1780316375.182612   38533 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780316375.183990   38533 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780316375.307023   38533 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780316378.512485   38533 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Pour l'entrainement de notre modèle de captionning nous avons utiliser le dataset COCO (2017) : https://cocodataset.org/#download

In [ ]:
# Chemins dataset COCO
image_folder = os.path.abspath("../../dataset/dataset_coco/train2017/")
annotation_file = os.path.abspath("../../dataset/dataset_coco/annotations/captions_train2017.json")

# Dictionnaire :
# clé   = chemin image
# valeur = liste des captions associées
image_path_to_caption = collections.defaultdict(list)

# Chargement du JSON COCO
with open(annotation_file, 'r', encoding='utf-8') as f:
    annotations_json = json.load(f)

# Parcourir toutes les annotations
for annot in annotations_json['annotations']:
    image_id = annot['image_id']
    caption = annot['caption']

    # Les images COCO sont nommées :
    # 000000000001.jpg
    image_filename = f"{image_id:012d}.jpg"

    image_path = os.path.join(image_folder, image_filename)

    # Vérifier existence
    if not os.path.exists(image_path):
        continue

    # Ajouter tokens début/fin
    caption = "<start> " + caption + " <end>"

    image_path_to_caption[image_path].append(caption)

# Liste des chemins d'images
image_paths = list(image_path_to_caption.keys())

# Limiter le dataset pour le workshop
train_image_paths = image_paths[:2000]

# Construire captions + vecteur images
train_captions = []
img_name_vector = []

for image_path in train_image_paths:
    # On récupère la liste des légendes (captions) associées à cette image
    caption_list = image_path_to_caption[image_path]

    # On ajoute toutes les captions de cette image à la liste globale des captions d'entraînement
    train_captions.extend(caption_list)

    # Dupliquer le chemin autant de fois qu'il y a de captions
    img_name_vector.extend([image_path] * len(caption_list))

# Affichage
print("train images:", len(train_image_paths))
print("train captions:", len(train_captions))
print("img_name_vector:", len(img_name_vector))

# Exemple
if len(train_image_paths) > 0:
    print("\nExample image:", train_image_paths[0])
    print("Example caption:", image_path_to_caption[train_image_paths[0]][0])

train images: 2000
train captions: 10010
img_name_vector: 10010

Example image: /home/charly/Documents/CESI_cours/A5/Option/projet-leyenda/dataset/dataset_coco/train2017/000000203564.jpg
Example caption: <start> A bicycle replica with a clock as the front wheel. <end>


# Benchmark des modèles proposés par Keras :

Le workshop 7 nous a présenté le modèle InceptionV3 mise à disposition par Keras. Après avoir fait des tests avec ce modèle nous nous sommes rendu compte que les résultats obtenus étaient médiocre. Nous avons donc mené des recherches afin de trouver le meilleur proposé par Keras. Sur le site officiel de Keras (https://keras.io/api/applications/) nous pouvons retrouver le benchmark suivant. Après comparatif nous avons décider d'utiliser le modèle EfficientNetV2M qui représente le meilleur compromis performance/temps d'entrainement.

| Model | Size (MB) | Top-1 Accuracy | Top-5 Accuracy | Parameters | Depth | Time (ms) per inference step (CPU) | Time (ms) per inference step (GPU) |
|---------|---------:|--------------:|--------------:|-----------:|------:|-----------------------------------:|-----------------------------------:|
| Xception | 88 | 79.0% | 94.5% | 22.9M | 81 | 109.4 | 8.1 |
| VGG16 | 528 | 71.3% | 90.1% | 138.4M | 16 | 69.5 | 4.2 |
| VGG19 | 549 | 71.3% | 90.0% | 143.7M | 19 | 84.8 | 4.4 |
| ResNet50 | 98 | 74.9% | 92.1% | 25.6M | 107 | 58.2 | 4.6 |
| ResNet50V2 | 98 | 76.0% | 93.0% | 25.6M | 103 | 45.6 | 4.4 |
| ResNet101 | 171 | 76.4% | 92.8% | 44.7M | 209 | 89.6 | 5.2 |
| ResNet101V2 | 171 | 77.2% | 93.8% | 44.7M | 205 | 72.7 | 5.4 |
| ResNet152 | 232 | 76.6% | 93.1% | 60.4M | 311 | 127.4 | 6.5 |
| ResNet152V2 | 232 | 78.0% | 94.2% | 60.4M | 307 | 107.5 | 6.6 |
| InceptionV3 | 92 | 77.9% | 93.7% | 23.9M | 189 | 42.2 | 6.9 |
| InceptionResNetV2 | 215 | 80.3% | 95.3% | 55.9M | 449 | 130.2 | 10.0 |
| MobileNet | 16 | 70.4% | 89.5% | 4.3M | 55 | 22.6 | 3.4 |
| MobileNetV2 | 14 | 71.3% | 90.1% | 3.5M | 105 | 25.9 | 3.8 |
| DenseNet121 | 33 | 75.0% | 92.3% | 8.1M | 242 | 77.1 | 5.4 |
| DenseNet169 | 57 | 76.2% | 93.2% | 14.3M | 338 | 96.4 | 6.3 |
| DenseNet201 | 80 | 77.3% | 93.6% | 20.2M | 402 | 127.2 | 6.7 |
| NASNetMobile | 23 | 74.4% | 91.9% | 5.3M | 389 | 27.0 | 6.7 |
| NASNetLarge | 343 | 82.5% | 96.0% | 88.9M | 533 | 344.5 | 20.0 |
| EfficientNetB0 | 29 | 77.1% | 93.3% | 5.3M | 132 | 46.0 | 4.9 |
| EfficientNetB1 | 31 | 79.1% | 94.4% | 7.9M | 186 | 60.2 | 5.6 |
| EfficientNetB2 | 36 | 80.1% | 94.9% | 9.2M | 186 | 80.8 | 6.5 |
| EfficientNetB3 | 48 | 81.6% | 95.7% | 12.3M | 210 | 140.0 | 8.8 |
| EfficientNetB4 | 75 | 82.9% | 96.4% | 19.5M | 258 | 308.3 | 15.1 |
| EfficientNetB5 | 118 | 83.6% | 96.7% | 30.6M | 312 | 579.2 | 25.3 |
| EfficientNetB6 | 166 | 84.0% | 96.8% | 43.3M | 360 | 958.1 | 40.4 |
| EfficientNetB7 | 256 | 84.3% | 97.0% | 66.7M | 438 | 1578.9 | 61.6 |
| EfficientNetV2B0 | 29 | 78.7% | 94.3% | 7.2M | - | - | - |
| EfficientNetV2B1 | 34 | 79.8% | 95.0% | 8.2M | - | - | - |
| EfficientNetV2B2 | 42 | 80.5% | 95.1% | 10.2M | - | - | - |
| EfficientNetV2B3 | 59 | 82.0% | 95.8% | 14.5M | - | - | - |
| EfficientNetV2S | 88 | 83.9% | 96.7% | 21.6M | - | - | - |
| EfficientNetV2M | 220 | 85.3% | 97.4% | 54.4M | - | - | - |
| EfficientNetV2L | 479 | 85.7% | 97.5% | 119.0M | - | - | - |
| ConvNeXtTiny | 109.42 | 81.3% | - | 28.6M | - | - | - |
| ConvNeXtSmall | 192.29 | 82.3% | - | 50.2M | - | - | - |
| ConvNeXtBase | 338.58 | 85.3% | - | 88.5M | - | - | - |
| ConvNeXtLarge | 755.07 | 86.3% | - | 197.7M | - | - | - |
| ConvNeXtXLarge | 1310 | 86.7% | - | 350.1M | - | - | - |

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # désactiver l'utilisation du GPU

# Contrôle le parallélisme CPU de TensorFlow afin de limiter le nombre de threads utilisés lors de l'exécution des opérations
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Téléchargement du modèle EfficientNetV2M pré-entraîné ImageNet
image_model = tf.keras.applications.EfficientNetV2M(
    include_top=False,
    weights="imagenet",
    input_shape=(480, 480, 3)
)

# Entrée du modèle
new_input = image_model.input

# Dernière couche convolutionnelle
hidden_layer = image_model.output

# Modèle extracteur de features
image_features_extract_model = tf.keras.Model(inputs=new_input, outputs=hidden_layer)

def load_image(image_path):
    # Lecture fichier
    img = tf.io.read_file(image_path)

    # Décodage image
    img = tf.image.decode_jpeg(img, channels=3)

    # Resize pour EfficientNetV2M (standard Keras)
    img = tf.image.resize(img, (480, 480))

    # IMPORTANT : preprocessing EfficientNetV2
    img = tf.keras.applications.efficientnet_v2.preprocess_input(img)

    return img, image_path


# On récupère uniquement les chemins d'images uniques (suppression des doublons)
encode_train = sorted(set(img_name_vector))

# Affiche le nombre total d'images uniques utilisées pour l'encodage
print("Nombre d'images uniques :", len(encode_train))

# Création d'un dataset TensorFlow à partir des chemins d'images
image_dataset = tf.data.Dataset.from_tensor_slices(tf.constant(encode_train))

# Application de la fonction de prétraitement "load_image" à chaque image
# num_parallel_calls=tf.data.AUTOTUNE permet d'optimiser les performances en parallèle
# puis regroupement des données par batch de 8 images
image_dataset = image_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(8)

for img_batch, path_batch in tqdm(image_dataset):
    # Features CNN
    batch_features = image_features_extract_model(img_batch)

    # (batch, H, W, C)
    # -> (batch, H*W, C)
    batch_features = tf.reshape(batch_features, (batch_features.shape[0], -1, batch_features.shape[-1]))

    # Sauvegarde individuelle
    for bf, p in zip(batch_features, path_batch):

        path_of_feature = p.numpy().decode("utf-8")

        # image.jpg.npy
        np.save(path_of_feature, bf.numpy())

print("Extraction terminée.")

E0000 00:00:1780316775.615439   38533 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Nombre d'images uniques : 2000


100%|██████████| 250/250 [31:43<00:00,  7.61s/it]

Extraction terminée.


**Pré-traitement des annotations**

In [ ]:
# Trouver la taille maximale 
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

# Chosir les 5000 mots les plus frequents du vocabulaire
top_k = 5000

# La classe Tokenizer permet de faire du pre-traitement de texte pour reseau de neurones 
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=top_k,
    oov_token="<unk>",
    filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '
)

# Construit un vocabulaire en se basant sur la liste train_captions
tokenizer.fit_on_texts(train_captions)

# Créer le token qui sert à remplir les annotations pour egaliser leurs longueur
tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'

# Creation des vecteurs(liste de token entiers) à partir des annotations (liste de mots)
train_seqs = tokenizer.texts_to_sequences(train_captions)

# Remplir chaque vecteur à jusqu'à la longueur maximale des annotations
cap_vector = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')

# Calcule la longueur maximale qui est utilisée pour stocker les poids d'attention 
# Elle servira plus tard pour l'affichage lors de l'évaluation
max_length = calc_max_length(train_seqs)

<>:12: SyntaxWarning: invalid escape sequence '\]'
<>:12: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_38533/1272394845.py:12: SyntaxWarning: invalid escape sequence '\]'
  filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '


# 1. Formation du jeu d'entrainement et de test 
Vous devez ensuite séparer le dataset en deux partie : un jeu d'entrainement et un jeu de test. Le code qui effectue ces opérations est détaillé dans la cellule suivante.

In [ ]:
img_to_cap_vector = collections.defaultdict(list)

# Creation d'un dictionnaire associant les chemins des images avec (fichier .npy) aux annotationss
# Les images sont dupliquées car il y a plusieurs annotations par image
print(len(img_name_vector), len(cap_vector))
for img, cap in zip(img_name_vector, cap_vector):
    img_to_cap_vector[img].append(cap)

"""
Création des datasets de formation et de validation en utilisant 
un fractionnement 80-20 de manière aléatoire
""" 
# clés des images
img_keys = list(img_to_cap_vector.keys())

# shuffle pour rester en python list
img_keys = np.array(img_keys)
np.random.shuffle(img_keys)

# Diviser des indices en entrainement et test
slice_index = int(len(img_keys) * 0.8)

img_name_train_keys = img_keys[:slice_index]
img_name_val_keys = img_keys[slice_index:]

"""
Les jeux d'entrainement et de tests sont sous forme
de listes contenants les mappings :(image prétraitée ---> jeton d'annotation(mot) )
"""

# Boucle pour construire le jeu d'entrainement
img_name_train = []
cap_train = []
for imgt in img_name_train_keys:
    imgt = str(imgt)
    capt_list = img_to_cap_vector[imgt]
    capt_len = len(capt_list)
    # Duplication des images en le nombre d'annotations par image
    img_name_train.extend([imgt] * capt_len)
    cap_train.extend(capt_list)

# Boucle pour construire le jeu de test
img_name_val = []
cap_val = []
for imgv in img_name_val_keys:
    imgv = str(imgv)
    capv_list = img_to_cap_vector[imgv]
    capv_len = len(capv_list)
    # Duplication des images en le nombre d'annotations par image
    img_name_val.extend([imgv] * capv_len)
    cap_val.extend(capv_list)

len(img_name_train), len(cap_train), len(img_name_val), len(cap_val)

10010 10010


(8007, 8007, 2003, 2003)

Création d’un jeu de données d’entrainement représenté par une instance [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) partant du jeu de données de base (les noms des fichiers et des annotations du jeu d’entrainement). La classe `tf.data.Dataset` sert à représenter des jeu de données volumineux et facilitent les prétraitements de ceux-ci.

In [ ]:
# N'hésitez pas à modifier ces paramètres en fonction de votre machine
BATCH_SIZE = 32 # taille du batch
BUFFER_SIZE = 1000 # taille du buffer pour melanger les donnees
embedding_dim = 256
units = 512 # Taille de la couche caché dans le RNN
vocab_size = top_k + 1
num_steps = len(img_name_train) // BATCH_SIZE

# La forme du vecteur extrait à partir d'EfficientNetV2M est (225, 1280)
# Les deux variables suivantes representent la forme de ce vecteur
features_shape = 1280
attention_features_shape = 225

# Fonction qui charge les fichiers numpy des images prétraitées
def map_func(img_name, cap):
    img_tensor = np.load(img_name.decode('utf-8')+'.npy')
    return img_tensor, cap

# Creation d'un dataset de "Tensor"s (sert à representer de grands dataset)
# Le dataset est cree a partir de "img_name_train" et "cap_train"
dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))

# L'utilisation de map permet de charger les fichiers numpy (possiblement en parallèle)
dataset = dataset.map(lambda item1, item2: tf.numpy_function(
          map_func, [item1, item2], [tf.float32, tf.int32]),
          num_parallel_calls=tf.data.experimental.AUTOTUNE)

# Melanger les donnees et les diviser en batchs
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# 2. Le modèle

Par rapport au modèle, la dernière couche de caractéristiques d’**EfficientNetV2M** produit généralement un tenseur de dimension `(7, 7, 1280)` avant les couches de classification finales. Ce tenseur est ensuite remodelé sous la forme `(49, 1280)` lors de son stockage sur le disque afin de conserver l’information spatiale extraite par le réseau convolutif. Ce vecteur de caractéristiques est ensuite transmis à l’encodeur CNN, composé d’une couche entièrement connectée permettant de projeter les caractéristiques visuelles dans un espace latent adapté à la génération de texte. Le RNN utilise alors cette représentation visuelle pour prédire séquentiellement les mots de l’annotation associée à l’image. Cette approche permet de combiner la capacité d’**EfficientNetV2M** à extraire des caractéristiques visuelles riches avec la capacité du RNN à modéliser les dépendances temporelles entre les mots afin de générer des descriptions cohérentes et pertinentes. L’image ci-dessous illustre un exemple simplifié d’une architecture d’annotation d’images reposant sur l’association d’un CNN pour l’extraction de caractéristiques visuelles et d’un RNN pour la génération de légendes.

Dans une tâche de *image captioning*, **EfficientNetV2M** est utilisé comme extracteur de caractéristiques visuelles. L’image d’entrée est d’abord prétraitée puis passée à travers les différentes couches convolutives du réseau, qui apprennent à détecter progressivement des motifs de plus en plus complexes (bords, textures, objets, scènes). La sortie de la dernière couche de caractéristiques, de dimension typique `(7, 7, 1280)`, est ensuite transformée en une séquence de vecteurs représentant les différentes régions de l’image. Ces vecteurs sont projetés dans un espace latent par un encodeur puis transmis à un décodeur séquentiel, généralement un réseau récurrent (LSTM/GRU) ou un transformeur. À partir de cette représentation visuelle, le décodeur génère la légende mot par mot en prédisant à chaque étape le terme le plus probable en fonction des caractéristiques de l’image et des mots déjà produits. Ainsi, EfficientNetV2M fournit une représentation compacte et riche du contenu visuel, tandis que le décodeur convertit cette information en une description textuelle cohérente de l’image.

**L'encodeur CNN :**

L'encodeur CNN a pour rôle de **projeter les caractéristiques visuelles extraites par EfficientNetV2M dans un espace de représentation adapté au décodeur**. Comme les images ont déjà été transformées en vecteurs de caractéristiques riches par le réseau convolutif pré-entraîné, cet encodeur n'effectue pas de nouvelles opérations de convolution. Il applique simplement une **couche dense (`Dense`)** afin de réduire ou d'adapter la dimension des caractéristiques à la taille de l'espace d'embedding utilisée par le modèle de génération de texte. La fonction d'activation **ReLU** introduit ensuite une non-linéarité permettant d'améliorer la capacité de représentation du modèle. La sortie obtenue est alors transmise au décodeur, qui s'appuie sur ces informations visuelles pour générer la légende de l'image mot par mot.

In [ ]:
class CNN_Encoder(tf.keras.Model):
    # Comme les images sont déjà prétraités par EfficientNetV2M est représenté sous forme compacte
    # L'encodeur CNN ne fera que transmettre ces caractéristiques à une couche dense
    def __init__(self, embedding_dim):
        super(CNN_Encoder, self).__init__()
        # forme après fc == (batch_size, 64, embedding_dim)
        self.fc = tf.keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

**Le mécanisme d'attention :**

Le mécanisme d’attention ressemble beaucoup à une cellule [RNN classique]( https://fr.wikipedia.org/wiki/R%C3%A9seau_de_neurones_r%C3%A9currents), mais avec quelques différences. La partie de l’attention a en entrée la représentation de l’image prétraitée retournée par le CNN ainsi que la valeur courante de l’état cachée du GRU, et en sortie le **vecteur du contexte** qui reflète les caractéristiques les plus importantes de l’image. Une étape intermédiaire pour calculer ce vecteur consiste à calculer les **poids d’attention** qui représentent l’importance de chaque position de l’image (il y en a 64) dans la prédiction de son annotation.

La représentation de l’image donnée en entrée est transformée au début de la même manière que pour le CNN en la passant à une couche dense de taille `units`. De même, l’état caché est aussi passé à une couche dense de taille `units`. La nouvelle représentation de l’image est ensuite additionnée à l’état caché puis passée à une fonction d’activation de type [`tanh`](https://fr.wikipedia.org/wiki/Tangente_hyperbolique) comme pour les cellules classiques de RNN. À ce niveau-là, on aura une représentation des données de taille `64xunits` contenant un mélange d’informations sur l’image et sur le texte de l’annotation. Un score est ensuite associé à chacune des positions en passant cette représentation à une couche dense. Ces scores sont normalisés avec une couche softmax pour produire le vecteur des **poids d’attention**. 

Finalement, chaque caractéristique de la représentation de l’image en entrée sera multipliée (pondérée) par le vecteur d’attention. Après quoi, on prend la somme de chaque caractéristique le long des positions (les lignes de la représentation) pour former le **vecteur du contexte**.

De façon globale, on peut dire que le vecteur d’attention dépend de scores qui sont appris à partir d’une représentation spatiale et textuelle de l’image. Ce vecteur d’attention renvoie la pertinence de chaque position et sert à calculer le vecteur du contexte qui nous donnera l’importance des caractéristiques de l’image.

In [ ]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # On ajoute une dimension temporelle à hidden pour pouvoir le combiner avec features
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        # Calcul de la couche d'attention :
        # combinaison des features et de l'état caché via des couches denses W1 et W2,
        # puis application d'une non-linéarité tanh
        attention_hidden_layer = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))

        # Projection de la représentation combinée vers un score d'attention (alignement)
        score = self.V(attention_hidden_layer)

        # Normalisation des scores pour obtenir des poids d'attention (somme = 1)
        attention_weights = tf.nn.softmax(score, axis=1)

        # Application des poids d'attention aux features pour obtenir une représentation pondérée
        context_vector = attention_weights * features

        # Agrégation des informations pondérées sur toutes les régions de l'image
        context_vector = tf.reduce_sum(context_vector, axis=1)
        
        return context_vector, attention_weights

**Le décodeur RNN :**

Le rôle du décodeur RNN est d’utiliser la représentation prétraitée de l’image de prédire sa légende mot par mot. Ce RNN à une seule cellule de type [GRU]( https://en.wikipedia.org/wiki/Gated_recurrent_unit). Le GRU a un état caché qui représente la mémoire des derniers éléments vu par celui-ci. Le GRU met à jour son état avant de le retourner, pour cela il utilise certains mécanismes de mémorisation qui sont assez sophistiqués.

Le décodeur est structuré comme suit, à chaque appel du RNN, le mot courant ainsi que la représentation de l’image et l’état caché du GRU sont donnés en entrée du RNN. Comme les mots sont représentés par des entiers, on doit faire passer ceux-ci par une couche dite [embedding layer]( https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) qui se chargera de calculer une représentation vectorielle de taille `output_dim` partant du nombre représentant le mot. 

À côté de ça, le mécanisme d’attention fournit un vecteur représentant **le contexte** de l’image c-à-d un vecteur qui nous renseigne sur les caractéristiques dominantes de l’image. Ce vecteur est calculé par un appel du mécanisme d’attention en lui fournissant en entrée les caractéristiques de l’image encodées par le CNN ainsi que l’état caché du GRU qui résume l’historique des mots vues par le RNN jusqu’à présent. 

Ensuite, le mot courant et le contexte sont concaténée pour former le vecteur d’entrée du GRU qui à son tour calcule l’état à l’étape suivante. Cet état est passée par une couche dense de taille `units` puis la sortie de cette couche est passée à une autre couche dense de taille `vocab_size` qui retourne le score associé à chaque mot du vocabulaire afin de prédire le mot suivant.

In [10]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super(RNN_Decoder, self).__init__()
        self.units = units

        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        self.gru = tf.keras.layers.GRU(self.units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

        #Couche dense qui aura pour entrée la sortie du GRU
        self.fc1 = tf.keras.layers.Dense(self.units)

        # Dernière couche dense
        self.fc2 = tf.keras.layers.Dense(vocab_size)

        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        # L'attention est defini par un modèle a part
        context_vector, attention_weights = self.attention(features, hidden)

        # Passage du mot courant à la couche embedding
        x = self.embedding(x)

        # Concaténation
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        # Passage du vecteur concaténé à la gru
        output, state = self.gru(x)

        # Couche dense
        y = self.fc1(output)

        y = tf.reshape(y, (-1, x.shape[2]))

        # Couche dense
        y = self.fc2(y)

        return y, state, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

**Combiner la partie encodeur et décodeur :**

Vous devez compléter la partie du code pour la création de l'encodeur et du décodeur.

In [11]:
# Création de l'encodeur
encoder = CNN_Encoder(embedding_dim)

# Création du décodeur
decoder = RNN_Decoder(embedding_dim, units, vocab_size)

In [ ]:
# Optimiseur ADAM
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# La fonction de perte
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    # Création d'un masque pour ignorer les tokens de padding (valeur 0)
    mask = tf.math.logical_not(tf.math.equal(real, 0))

    # Calcul de la perte entre les vraies valeurs (real) et les prédictions (pred)
    loss_ = loss_object(real, pred)

    # Conversion du masque en type compatible avec la perte calculée
    mask = tf.cast(mask, dtype=loss_.dtype)

    # Application du masque : on annule la perte des tokens de padding
    loss_ *= mask

    # Calcul de la moyenne de la perte sur tous les éléments valides
    return tf.reduce_mean(loss_)

Pour garder la trace de votre apprentissage et la sauvegarder, vous pouvez utiliser la classe [`tf.train.Checkpoint`](https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint).

In [13]:
checkpoint_path = "./checkpoints/train"
ckpt = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer = optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

Initialisation de l'époque de début d’entrainement dans `start_epoch`. La classe `tf.train.Checkpoint` permet de poursuivre l’entrainement là ou vous l’avez laissé s’il avait été interrompu auparavant.

In [14]:
start_epoch = 0
if ckpt_manager.latest_checkpoint:
    start_epoch = int(ckpt_manager.latest_checkpoint.split('-')[-1])
    # Restaurer le dernier checkpoint dans checkpoint_path
    ckpt.restore(ckpt_manager.latest_checkpoint)

# 3. Entrainement et test

On a ensuite implémenté les fonctions `train_step` et `evaluate`.

* La fonction `train_step` représente une étape d’entraînement du réseau. Elle commence par l’encodage du vecteur pré-calculé par EfficientNetV2M (ou InceptionV3 selon l’architecture utilisée). La sortie obtenue est ensuite transmise au décodeur, qui se charge de générer l’annotation mot par mot. On a également intégré dans cette fonction la boucle permettant de prédire chaque mot de la séquence ainsi que le calcul de la perte associée.

* La fonction `evaluate` permet d’évaluer les performances du modèle sur le jeu de test. Son fonctionnement est similaire à celui de `train_step`, à la différence qu’elle ne contient pas la partie liée au calcul et à l’optimisation de la fonction de perte, puisqu’il ne s’agit pas ici d’entraîner le réseau mais uniquement de mesurer ses performances.

Enfin, on a implémenté la partie du code dédiée à l’entraînement et au test du modèle. L’entraînement est réalisé par batch d’images afin d’optimiser l’utilisation des ressources et de stabiliser l’apprentissage.

## 3.1. Entrainement

La fonction qui permet d'achever une étape d'entrainement sur un batch d'images est `train_step`. La fonction a en entrée un batch d'images prétraitées ainsi que leurs annotations et retourne la perte associée à ce batch. 

L'état caché de la partie RNN est initialisé ainsi que le mot de départ avec le token de début. Les caractéristiques de l'image sont ensuite extraites par l’encodeur. Après cela, on parcourt le batch mot par mot pour prédire le mot suivant à l'aide du décodeur. Le décodeur utilise l'état caché, les caractéristiques de l'image ainsi que le mot précédent pour prédire le mot courant. Le décodeur met à jour l'état caché et le retourne ainsi que les prédictions du batch. La perte est calculée à partir des prédictions retournées par le décodeur et les annotations associées au batch.

Finalement, la perte globale ainsi que le gradient sont calculés et le réseau est mis à jour.

In [ ]:
loss_plot = []
@tf.function
def train_step(img_tensor, target):
    loss = 0

    # Initialisation de l'état caché pour chaque batch
    hidden = decoder.reset_state(batch_size=target.shape[0])
    
    # Initialiser l'entrée du décodeur
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * target.shape[0], 1)
    
    with tf.GradientTape() as tape: # Offre la possibilité de calculer le gradient du loss
        features = encoder(img_tensor)

        for i in range(1, target.shape[1]):
            # Prédiction des i'èmes mot du batch avec le décodeur
            predictions, hidden, _ = decoder(dec_input, features, hidden)
            loss += loss_function(target[:, i], predictions)

            # Le mot correct à l'étap i est donné en entrée à l'étape (i+1)
            dec_input = tf.expand_dims(target[:, i], 1)

    # Calcul de la perte moyenne sur toute la séquence de mots
    # On divise la perte totale par la longueur de la séquence cible
    total_loss = (loss / int(target.shape[1]))

    # Récupération de toutes les variables entraînables du modèle
    # On combine les variables de l'encodeur et du décodeur
    trainable_variables = encoder.trainable_variables + decoder.trainable_variables

    # Calcul des gradients de la perte par rapport à toutes les variables entraînables
    # GradientTape enregistre les opérations pour pouvoir effectuer la rétropropagation
    gradients = tape.gradient(loss, trainable_variables)

    # Application des gradients calculés pour mettre à jour les poids du modèle
    # L'optimiseur ajuste les paramètres de l'encodeur et du décodeur
    optimizer.apply_gradients(zip(gradients, trainable_variables))

    return loss, total_loss

Le code global contenant la boucle d'entrainement est présenté ci-dessous. Cette boucle parcours le jeu de données d'entrainement batch par batch et entraine le réseaux avec ceux-ci. 40 époques sont suffisantes pour EfficientNet pour converger vers un résultat pertinant. 

In [ ]:
EPOCHS = 40

for epoch in range(start_epoch, EPOCHS):
    start = time.time()
    total_loss = 0
    
    for (batch, (img_tensor, target)) in enumerate(dataset):
        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += t_loss

        if batch % 100 == 0:
            print ('Epoch {} Batch {} Loss {:.4f}'.format(
              epoch + 1, batch, batch_loss.numpy() / int(target.shape[1])))
    # sauvegarde de la perte
    loss_plot.append(total_loss / num_steps)

    if epoch % 5 == 0:
        ckpt_manager.save()

    print ('Epoch {} Loss {:.6f}'.format(epoch + 1, total_loss/num_steps))
    print ('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

# Affichage de la courbe d'entrainement
plt.plot(loss_plot)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Plot')
plt.show()

# 3.2 Test
La fonction qui permet d'achever une étape d'evaluation pour le test est dans la cellule suivante. 

In [ ]:
def evaluate(image):
    # matrice pour stocker les poids d'attention
    attention_plot = np.zeros((max_length, attention_features_shape))

    # initialisation de l'état caché du décodeur
    hidden = decoder.reset_state(batch_size=1)

    # chargement et mise en batch de l'image
    temp_input = tf.expand_dims(load_image(image)[0], 0)

    # extraction des features CNN
    img_tensor_val = image_features_extract_model(temp_input)

    # reshape en séquence
    img_tensor_val = tf.reshape(img_tensor_val, (img_tensor_val.shape[0], -1, img_tensor_val.shape[3]))

    # passage dans l'encodeur CNN
    features = encoder(img_tensor_val)

    # token de départ pour le décodeur
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)

    result = []

    for i in range(max_length):
        # prédiction du prochain mot + attention
        predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

        # sauvegarde des poids d'attention
        attention_plot[i] = tf.reshape(attention_weights, (-1, )).numpy()

        # échantillonnage du mot prédit
        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()

        # conversion id → mot et ajout au résultat
        result.append(tokenizer.index_word[predicted_id])

        if tokenizer.index_word[predicted_id] == '<end>':
            return result, attention_plot

        # entrée du prochain pas de temps
        dec_input = tf.expand_dims([predicted_id], 0)

    # troncature du plot à la longueur réelle
    attention_plot = attention_plot[:len(result), :]
    
    return result, attention_plot

# Fonction permettant la représentation de l'attention au niveau de l'image
def plot_attention(image, result, attention_plot):
    temp_image = np.array(Image.open(image))

    fig = plt.figure(figsize=(10, 10))

    len_result = len(result)
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(len_result//2, len_result//2, l+1)
        ax.set_title(result[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap='gray', alpha=0.6, extent=img.get_extent())

    plt.tight_layout()
    plt.show()

In [17]:
# Affichage d'une image et de sa prédiction propre
rid = np.random.randint(0, len(img_name_val))
image_path = img_name_val[rid]

# Caption réelle (nettoyée)
real_caption = ' '.join(
    [tokenizer.index_word[i] for i in cap_val[rid] if i not in [0]]
).replace('<start>', '').replace('<end>', '').strip()

# Prédiction du modèle
result, _ = evaluate(image_path)

# Nettoyage de la prédiction
pred_caption = ' '.join(
    [word for word in result if word not in ['<start>', '<end>']]
).strip()

# Chargement image
img = Image.open(image_path)

# Affichage
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.title(pred_caption, fontsize=12)
plt.show()

print("Real caption:", real_caption)
print("Predicted caption:", pred_caption)

KeyError: np.int64(4816)

## 3.3. Sauvegarder le modèle

Sauvegarde :
- les poids de l’encoder,
- les poids du decoder,
- le tokenizer,
- la config nécessaire pour recharger le modèle plus tard.

In [ ]:
save_dir = "../../models/livrable3"

# Création du dossier si nécessaire
os.makedirs(save_dir, exist_ok=True)

# Sauvegarde des poids
encoder.save_weights(os.path.join(save_dir, "encoder.weights.h5"))
decoder.save_weights(os.path.join(save_dir, "decoder.weights.h5"))

# Sauvegarde du tokenizer
with open(os.path.join(save_dir, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# Sauvegarde des paramètres utiles
config = {
    "embedding_dim": embedding_dim,
    "units": units,
    "vocab_size": vocab_size,
    "max_length": max_length,
    "features_shape": features_shape,
    "attention_features_shape": attention_features_shape
}

with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump(config, f)

print("Modèle sauvegardé dans :", save_dir)